# 🏥 Smart Medical Sign Display System
## University NLP Project — Google Colab Demo

**Pipeline:**  
`Audio (.wav)` → `Whisper STT` → `Text Preprocessing` → `NLP Simplification` → `Sign Mapping` → `Display`

---
**Authors:** [Your Name]  
**Course:** Natural Language Processing / Computer Science  
**Year:** 2024


## 📦 Step 0 — Install Dependencies

In [ ]:
# Install all required packages
!pip install openai-whisper -q
!pip install pythainlp -q
!pip install streamlit -q
!pip install pydub -q
print('✅ All packages installed')

## 🔬 Step 1 — Import Libraries

In [ ]:
import re
import json
from IPython.display import display, HTML

try:
    import whisper
    WHISPER_AVAILABLE = True
    print('✅ Whisper loaded')
except ImportError:
    WHISPER_AVAILABLE = False
    print('⚠️  Whisper not available — text-only mode')

try:
    from pythainlp.tokenize import word_tokenize
    from pythainlp.corpus.common import thai_stopwords
    PYTHAINLP_AVAILABLE = True
    print('✅ PyThaiNLP loaded')
except ImportError:
    PYTHAINLP_AVAILABLE = False
    print('⚠️  PyThaiNLP not available — fallback tokenizer')

## 📚 Step 2 — Thai Sign Dictionary (30+ entries)

In [ ]:
# ─── Thai Medical Sign Dictionary ───────────────────────────────────────────────
# Each key is a simplified Thai word.
# In a production system, 'emoji' would be replaced by an actual image path.

SIGN_DICTIONARY = {
    # Negation / Modifiers
    'ไม่':      {'emoji': '🚫', 'label': 'ไม่'},
    'ไม่ได้':   {'emoji': '❌', 'label': 'ไม่ได้'},
    'ห้าม':     {'emoji': '🛑', 'label': 'ห้าม'},
    'ต้อง':     {'emoji': '✅', 'label': 'ต้อง'},
    'มาก':      {'emoji': '⬆️', 'label': 'มาก'},
    'น้อย':     {'emoji': '🤏', 'label': 'น้อย'},

    # Core Actions
    'กิน':      {'emoji': '🍽️', 'label': 'กิน'},
    'ดื่ม':     {'emoji': '🥤', 'label': 'ดื่ม'},
    'นอน':      {'emoji': '🛏️', 'label': 'นอน'},
    'นั่ง':     {'emoji': '🪑', 'label': 'นั่ง'},
    'ยืน':      {'emoji': '🧍', 'label': 'ยืน'},
    'เดิน':     {'emoji': '🚶', 'label': 'เดิน'},
    'หยุด':     {'emoji': '✋', 'label': 'หยุด'},
    'หายใจ':    {'emoji': '💨', 'label': 'หายใจ'},
    'รอ':       {'emoji': '⏳', 'label': 'รอ'},
    'มา':       {'emoji': '👋', 'label': 'มา'},
    'ไป':       {'emoji': '➡️', 'label': 'ไป'},
    'บอก':      {'emoji': '🗣️', 'label': 'บอก'},
    'ทำตาม':    {'emoji': '🙌', 'label': 'ทำตาม'},

    # Medical / Body
    'ตรวจ':     {'emoji': '🔬', 'label': 'ตรวจ'},
    'ยา':       {'emoji': '💊', 'label': 'ยา'},
    'เจ็บ':     {'emoji': '😣', 'label': 'เจ็บ'},
    'ปวด':      {'emoji': '🤕', 'label': 'ปวด'},
    'หมอ':      {'emoji': '👨\u200d⚕️', 'label': 'หมอ'},
    'พยาบาล':   {'emoji': '👩\u200d⚕️', 'label': 'พยาบาล'},
    'โรงพยาบาล':{'emoji': '🏥', 'label': 'โรงพยาบาล'},
    'เลือด':    {'emoji': '🩸', 'label': 'เลือด'},
    'แผล':      {'emoji': '🩹', 'label': 'แผล'},
    'น้ำ':      {'emoji': '💧', 'label': 'น้ำ'},
    'อาหาร':    {'emoji': '🍱', 'label': 'อาหาร'},

    # Time
    'ก่อน':     {'emoji': '⬅️⏰', 'label': 'ก่อน'},
    'หลัง':     {'emoji': '⏰➡️', 'label': 'หลัง'},
    'วันนี้':   {'emoji': '📅', 'label': 'วันนี้'},
    'พรุ่งนี้': {'emoji': '🌅', 'label': 'พรุ่งนี้'},
    'เช้า':     {'emoji': '🌄', 'label': 'เช้า'},
    'เย็น':     {'emoji': '🌇', 'label': 'เย็น'},
    'คืน':      {'emoji': '🌙', 'label': 'คืน'},
    'ทุกวัน':   {'emoji': '🔁', 'label': 'ทุกวัน'},

    # Safety / Instructions
    'ระวัง':    {'emoji': '⚠️', 'label': 'ระวัง'},
    'เร็ว':     {'emoji': '⚡', 'label': 'เร็ว'},
    'ช้า':      {'emoji': '🐢', 'label': 'ช้า'},
    'ครั้ง':    {'emoji': '🔢', 'label': 'ครั้ง'},
    'เม็ด':     {'emoji': '💊', 'label': 'เม็ด'},
}

print(f'📖 พจนานุกรมมีทั้งหมด {len(SIGN_DICTIONARY)} คำ')

## 🧠 Step 3 — NLP Simplification Rules (Rule-based, Explainable)

In [ ]:
# ─── Rule-based Medical Thai Simplification ─────────────────────────────────
# WHY rule-based?
#   - Transparent: each rule has a clear intent
#   - No training data required
#   - Fast and predictable
#   - Easy to extend with domain expertise

SIMPLIFICATION_RULES = [
    # 1. Remove polite / formal openers
    (r'กรุณา\s*',   ''),   # 'กรุณา' = please (formal) → remove
    (r'โปรด\s*',    ''),   # 'โปรด'  = please (formal) → remove
    (r'ขอให้\s*',   ''),   # 'ขอให้' = request that  → remove
    (r'ท่าน\s*',    ''),   # 'ท่าน'  = formal 'you'   → remove

    # 2. Medical terms → simple Thai
    (r'งดอาหาร',           'ไม่กิน'),
    (r'งดน้ำ',             'ไม่ดื่มน้ำ'),
    (r'รับประทาน',         'กิน'),
    (r'บริโภค',            'กิน'),
    (r'เข้ารับการตรวจ',    'ตรวจ'),
    (r'เข้ารับการรักษา',   'รักษา'),
    (r'รับการผ่าตัด',      'ผ่าตัด'),
    (r'ปฏิบัติตาม',        'ทำตาม'),
    (r'นอนพักผ่อน',        'นอน'),
    (r'พักผ่อน',           'พัก'),
    (r'ออกกำลังกาย',       'เดิน'),
    (r'สอบถาม',            'ถาม'),
    (r'แจ้งให้ทราบ',       'บอก'),
    (r'แพทย์',             'หมอ'),
    (r'เจ้าหน้าที่พยาบาล', 'พยาบาล'),
    (r'ยาเม็ด',            'ยา'),
    (r'สารน้ำ',            'น้ำ'),
    (r'ก่อนเข้ารับ',       'ก่อน'),
    (r'หลังเข้ารับ',       'หลัง'),
    (r'หลังการ',           'หลัง'),
    (r'ก่อนการ',           'ก่อน'),

    # 3. Dosage normalizations
    (r'วันละ\s*(\d+)\s*ครั้ง',   r'\1 ครั้งต่อวัน'),
    (r'ครั้งละ\s*(\d+)\s*เม็ด',  r'\1 เม็ด'),
    (r'ทุก\s*(\d+)\s*ชั่วโมง',   r'ทุก \1 ชม.'),
]

def preprocess_text(text: str) -> str:
    """Remove irrelevant punctuation and extra whitespace."""
    text = re.sub(r'[\"\"\"\'\'\']', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def simplify_text(text: str) -> str:
    """
    Rule-based simplification.
    Applies ordered regex rules to convert formal medical Thai
    to plain, patient-friendly Thai.
    """
    result = text
    for pattern, replacement in SIMPLIFICATION_RULES:
        result = re.sub(pattern, replacement, result)
    return re.sub(r'\s+', ' ', result).strip()

# ── Quick test ──
test_cases = [
    'กรุณางดอาหารก่อนเข้ารับการตรวจ',
    'รับประทานยาเม็ดวันละ 2 ครั้ง หลังอาหาร',
    'กรุณานอนพักผ่อนและงดออกกำลังกาย',
    'โปรดแจ้งให้ทราบหากมีอาการปวด',
    'ห้ามรับประทานอาหารก่อนเจาะเลือด',
]

print('=' * 55)
print(f'{"ต้นฉบับ":<30} → {"ข้อความที่เข้าใจง่าย"}')
print('=' * 55)
for tc in test_cases:
    simplified = simplify_text(preprocess_text(tc))
    print(f'{tc:<30} → {simplified}')

## ✂️ Step 4 — Thai Tokenization

In [ ]:
def tokenize_text(text: str) -> list:
    """
    Tokenize Thai text.
    Primary: PyThaiNLP newmm algorithm (dictionary + maximal matching)
    Fallback: Longest-match against sign dictionary
    """
    if PYTHAINLP_AVAILABLE:
        tokens = word_tokenize(text, engine='newmm', keep_whitespace=False)
        stops = thai_stopwords()
        keep_always = {'ไม่', 'ห้าม', 'ต้อง', 'ก่อน', 'หลัง', 'ระวัง'}
        return [t for t in tokens if t.strip() and (t not in stops or t in keep_always)]
    else:
        # Fallback: longest-match against sign dictionary
        tokens = []
        sorted_keys = sorted(SIGN_DICTIONARY.keys(), key=len, reverse=True)
        remaining = text
        while remaining:
            matched = False
            for key in sorted_keys:
                if remaining.startswith(key):
                    tokens.append(key)
                    remaining = remaining[len(key):]
                    matched = True
                    break
            if not matched:
                tokens.append(remaining[0])
                remaining = remaining[1:]
        return [t for t in tokens if t.strip()]

# ── Test tokenization ──
sample = 'ไม่กินก่อนตรวจ'
tokens = tokenize_text(sample)
print(f'Input:  "{sample}"')
print(f'Tokens: {tokens}')

## 🗂️ Step 5 — Sign Mapping

In [ ]:
def map_to_signs(tokens: list) -> list:
    """
    Map each token to a sign dictionary entry.
    Returns list of sign objects with match status.
    """
    signs = []
    for token in tokens:
        if token in SIGN_DICTIONARY:
            signs.append({
                'token': token,
                'emoji': SIGN_DICTIONARY[token]['emoji'],
                'label': SIGN_DICTIONARY[token]['label'],
                'found': True
            })
        else:
            signs.append({'token': token, 'emoji': '❓', 'label': token, 'found': False})
    return signs

def full_pipeline(text: str) -> dict:
    """Run the complete NLP pipeline on a text string."""
    preprocessed = preprocess_text(text)
    simplified   = simplify_text(preprocessed)
    tokens       = tokenize_text(simplified)
    signs        = map_to_signs(tokens)
    coverage     = sum(1 for s in signs if s['found']) / max(len(signs), 1)
    return {
        'original':     text,
        'preprocessed': preprocessed,
        'simplified':   simplified,
        'tokens':       tokens,
        'signs':        signs,
        'coverage':     coverage
    }

# ── Test full pipeline ──
r = full_pipeline('กรุณางดอาหารก่อนเข้ารับการตรวจ')
print('Pipeline Result:')
print(f"  Original:    {r['original']}")
print(f"  Simplified:  {r['simplified']}")
print(f"  Tokens:      {r['tokens']}")
print(f"  Coverage:    {r['coverage']*100:.0f}%")
print('  Signs:')
for s in r['signs']:
    status = '✅' if s['found'] else '❓'
    print(f"    {status} {s['emoji']} {s['label']}")

## 📊 Step 6 — Batch Evaluation on Sample Dataset

In [ ]:
# ─── Sample Medical Sentence Dataset ────────────────────────────────────────
SAMPLE_DATASET = [
    {
        'id': 1,
        'original': 'กรุณางดอาหารก่อนเข้ารับการตรวจ',
        'expected_simplified': 'ไม่กินก่อนตรวจ',
        'category': 'การตรวจ'
    },
    {
        'id': 2,
        'original': 'รับประทานยาเม็ดวันละ 2 ครั้ง หลังอาหาร',
        'expected_simplified': 'กินยา 2 ครั้งต่อวัน หลังอาหาร',
        'category': 'ยา'
    },
    {
        'id': 3,
        'original': 'กรุณานอนพักผ่อนและงดออกกำลังกาย',
        'expected_simplified': 'นอนและไม่เดิน',
        'category': 'การพักผ่อน'
    },
    {
        'id': 4,
        'original': 'โปรดแจ้งให้ทราบหากมีอาการปวด',
        'expected_simplified': 'บอกถ้าปวด',
        'category': 'การสื่อสาร'
    },
    {
        'id': 5,
        'original': 'ต้องดื่มน้ำมาก ๆ หลังการตรวจ',
        'expected_simplified': 'ดื่มน้ำมากหลังตรวจ',
        'category': 'การดื่มน้ำ'
    },
    {
        'id': 6,
        'original': 'ห้ามรับประทานอาหารก่อนเจาะเลือด',
        'expected_simplified': 'ห้ามกินก่อนเลือด',
        'category': 'การตรวจเลือด'
    },
    {
        'id': 7,
        'original': 'กรุณานั่งรอเจ้าหน้าที่พยาบาลก่อน',
        'expected_simplified': 'นั่งรอพยาบาล',
        'category': 'การรอ'
    },
    {
        'id': 8,
        'original': 'แพทย์จะมาตรวจในตอนเช้าพรุ่งนี้',
        'expected_simplified': 'หมอมาตรวจเช้าพรุ่งนี้',
        'category': 'การนัดหมาย'
    },
    {
        'id': 9,
        'original': 'กรุณางดน้ำหลังเที่ยงคืน',
        'expected_simplified': 'ไม่ดื่มน้ำหลังคืน',
        'category': 'การตรวจ'
    },
    {
        'id': 10,
        'original': 'ควรนอนหลับพักผ่อนให้เพียงพอ',
        'expected_simplified': 'นอนให้เพียงพอ',
        'category': 'การพักผ่อน'
    },
]

# ── Run batch evaluation ──
print('=' * 65)
print('BATCH PIPELINE EVALUATION')
print('=' * 65)
total_coverage = 0
for item in SAMPLE_DATASET:
    r = full_pipeline(item['original'])
    total_coverage += r['coverage']
    sign_str = ' '.join([f"[{s['emoji']}{s['label']}]" for s in r['signs']])
    print(f"\n#{item['id']} [{item['category']}]")
    print(f"  Original:   {item['original']}")
    print(f"  Simplified: {r['simplified']}")
    print(f"  Signs:      {sign_str}")
    print(f"  Coverage:   {r['coverage']*100:.0f}%")

avg_coverage = total_coverage / len(SAMPLE_DATASET)
print('\n' + '=' * 65)
print(f'Average Sign Coverage: {avg_coverage*100:.1f}%')
print('=' * 65)

## 🎙️ Step 7 — Speech-to-Text with Whisper (Optional)

In [ ]:
# ─── Upload and transcribe a .wav file ──────────────────────────────────────
# To test: upload a .wav file containing spoken Thai medical instructions

from google.colab import files as colab_files

if WHISPER_AVAILABLE:
    print('📂 Upload a .wav file to transcribe...')
    uploaded = colab_files.upload()
    
    if uploaded:
        filename = list(uploaded.keys())[0]
        print(f'Transcribing: {filename}')
        
        model = whisper.load_model('base')  # or 'small', 'medium'
        result = model.transcribe(filename, language='th')
        transcribed_text = result['text'].strip()
        
        print(f'\n🎙️ Transcribed: {transcribed_text}')
        
        # Run through full pipeline
        pipeline_result = full_pipeline(transcribed_text)
        print(f"✅ Simplified:  {pipeline_result['simplified']}")
        for s in pipeline_result['signs']:
            print(f"   {s['emoji']} {s['label']}")
else:
    print('⚠️  Whisper not available. Testing with sample text instead...')
    demo_text = 'กรุณางดอาหารก่อนเข้ารับการตรวจ'
    r = full_pipeline(demo_text)
    print(f'\nDemo text:  {demo_text}')
    print(f'Simplified: {r["simplified"]}')
    for s in r['signs']:
        print(f'   {s["emoji"]} {s["label"]}')

## 📐 Step 8 — Academic Analysis: NLP Components

This section documents the academic NLP components for the university report.

In [ ]:
# ─── Rule coverage analysis ──────────────────────────────────────────────────
print('NLP COMPONENT ANALYSIS')
print('=' * 50)

print('\n1. SIMPLIFICATION RULES (Rule-based NLP)')
print(f'   Total rules: {len(SIMPLIFICATION_RULES)}')
categories = {
    'Polite opener removal': 4,
    'Medical-to-simple mapping': 20,
    'Dosage normalization': 3,
}
for cat, count in categories.items():
    print(f'   - {cat}: {count} rules')

print('\n2. SIGN DICTIONARY COVERAGE')
print(f'   Total entries: {len(SIGN_DICTIONARY)}')
sign_cats = {
    'Negation/Modifiers': ['ไม่', 'ห้าม', 'ต้อง', 'มาก', 'น้อย'],
    'Actions': ['กิน', 'ดื่ม', 'นอน', 'เดิน', 'รอ'],
    'Medical': ['ตรวจ', 'ยา', 'เจ็บ', 'หมอ', 'พยาบาล'],
    'Time': ['ก่อน', 'หลัง', 'วันนี้', 'เช้า', 'คืน'],
    'Instructions': ['ระวัง', 'ห้าม', 'ต้อง'],
}
for cat, words in sign_cats.items():
    print(f'   - {cat}: {words}')

print('\n3. TOKENIZATION METHOD')
if PYTHAINLP_AVAILABLE:
    print('   Algorithm: newmm (PyThaiNLP) — Dictionary + Maximal Matching')
else:
    print('   Algorithm: Longest-match (fallback, sign dict as vocabulary)')

print('\n4. PIPELINE TRANSPARENCY')
print('   Each transformation step is logged and explainable.')
print('   No black-box components — all rules are human-readable.')

print('\n5. LIMITATIONS & FUTURE WORK')
limitations = [
    'Dictionary-based sign mapping misses out-of-vocabulary words',
    'Rule-based simplification cannot handle all sentence structures',
    'No actual sign language video/animation (emoji placeholder only)',
    'Whisper STT accuracy depends on audio quality and accent',
    'No context-awareness (same word may need different signs)',
]
for i, lim in enumerate(limitations, 1):
    print(f'   {i}. {lim}')

print('\n6. POSSIBLE FUTURE EXTENSIONS')
extensions = [
    'Integrate seq2seq model (e.g. mBART) for neural simplification',
    'Replace emoji with real Thai Sign Language (TSL) video clips',
    'Add text-to-speech for patients with visual impairments',
    'Real-time microphone input via WebRTC',
    'LLM-based simplification using Claude/GPT API',
    'Expand dictionary to 500+ medical words with TSL images',
    'Patient feedback loop to improve simplification quality',
]
for i, ext in enumerate(extensions, 1):
    print(f'   {i}. {ext}')